# Multimodal GIF Generator — WAV Audio Input

Extracts **audio** and **text (transcript)** signals from a single `.wav` file and produces **two time-synchronised GIFs** that play at the original recording's speed:

| GIF | Content |
|-----|---------|
| `gif1_audio_waveform.gif` | Scrolling time-vs-amplitude waveform with rolling playhead, word-region shading, progress bar, and emotion badge |
| `gif2_text_display.gif` | Large-font word-by-word highlight with an inline mini-waveform strip, per-word fill bars, and a timeline ruler |

### Signal extraction pipeline
1. **Audio** — the original WAV is loaded at full resolution (48 kHz) for crisp waveform rendering; a 16 kHz mono copy is used for lightweight VAD processing.
2. **Transcript** — the RAVDESS filename encodes both the sentence (`01` → *"Kids are talking by the door"*, `02` → *"Dogs are sitting by the door"*) and rich metadata (emotion, actor, intensity). A built-in dictionary maps the code to the full sentence instantly — no model download required.
3. **Word timestamps** — short-time energy VAD (8 ms frames, 3 ms hop, 12 ms smoothing) finds energy peaks within the voiced region; midpoints between consecutive peaks become word boundaries, giving linguistically balanced segments.

> Both GIFs share the **same frame count and frame delay**, so they stay in perfect sync side-by-side.


## Cell 1 — Imports

In [ ]:
import os, io, math, subprocess, warnings, time
import numpy as np
import scipy.io.wavfile as wavfile
from scipy.signal import find_peaks
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

warnings.filterwarnings("ignore", category=wavfile.WavFileWarning)

print("All imports OK")
for name, obj in [("numpy", np), ("matplotlib", matplotlib)]:
    print(f"  {name:<12}  {obj.__version__}")


## Cell 2 — Configuration  *(edit here)*

In [ ]:
# ── Input ─────────────────────────────────────────────────────────────────────
WAV_PATH = "/mnt/user-data/uploads/03-01-01-01-01-01-01.wav"   # ← change this path

# ── Output ────────────────────────────────────────────────────────────────────
OUT_DIR  = "."
OUT_GIF1 = os.path.join(OUT_DIR, "gif1_audio_waveform.gif")
OUT_GIF2 = os.path.join(OUT_DIR, "gif2_text_display.gif")

# ── Temp resampled audio (for VAD) ────────────────────────────────────────────
WAV_16K  = "/tmp/_notebook_16k.wav"

# ── GIF parameters ────────────────────────────────────────────────────────────
GIF_FPS          = 30          # frames per second
FRAME_MS         = 33          # ms per GIF frame  (33 ≈ 30 fps)
OUTPUT_SIZE      = (640, 360)  # (width, height) of each GIF frame

# ── Visual palette ────────────────────────────────────────────────────────────
BG_COLOR        = "#0D1B2A"
ACCENT_COLOR    = "#38BDF8"
WAVE_COLOR      = "#34D399"
TEXT_FG_COLOR   = "#E2E8F0"
HIGHLIGHT_COLOR = "#FBBF24"   # amber — currently spoken word

print("Configuration loaded.")
print(f"  Input  : {WAV_PATH}")
print(f"  GIF1   : {OUT_GIF1}")
print(f"  GIF2   : {OUT_GIF2}")
print(f"  Speed  : {GIF_FPS} fps  ({FRAME_MS} ms/frame)")
print(f"  Size   : {OUTPUT_SIZE[0]}×{OUTPUT_SIZE[1]}")


## Cell 3 — RAVDESS filename dictionary

In [ ]:
# RAVDESS filename format:  MM-CC-EE-II-SS-RR-AA.wav
#   MM = Modality   01=full-AV  02=video-only  03=audio-only
#   CC = Channel    01=speech   02=song
#   EE = Emotion    01–08
#   II = Intensity  01=normal   02=strong
#   SS = Statement  01 or 02
#   RR = Repetition 01 or 02
#   AA = Actor      01–24  (odd=male  even=female)

RAVDESS_MODALITY  = {"01":"full-AV","02":"video-only","03":"audio-only"}
RAVDESS_CHANNEL   = {"01":"speech","02":"song"}
RAVDESS_EMOTION   = {
    "01":"neutral","02":"calm","03":"happy","04":"sad",
    "05":"angry","06":"fearful","07":"disgust","08":"surprised",
}
RAVDESS_INTENSITY = {"01":"normal","02":"strong"}
RAVDESS_STATEMENT = {
    "01": "Kids are talking by the door",
    "02": "Dogs are sitting by the door",
}
RAVDESS_EMO_COLOUR = {
    "neutral":"#94A3B8","calm":"#60A5FA","happy":"#4ADE80","sad":"#818CF8",
    "angry":"#F87171","fearful":"#FB923C","disgust":"#C084FC","surprised":"#F472B6",
}

def parse_ravdess(path):
    """
    Parse a RAVDESS filename and return a metadata dict.
    Returns None if the filename does not match the RAVDESS pattern.
    """
    stem  = os.path.splitext(os.path.basename(path))[0]
    parts = stem.split("-")
    if len(parts) != 7:
        return None
    mod, ch, emo, inten, stmt, rep, actor = parts
    actor_id = int(actor)
    return {
        "modality"  : RAVDESS_MODALITY.get(mod,   mod),
        "channel"   : RAVDESS_CHANNEL.get(ch,     ch),
        "emotion"   : RAVDESS_EMOTION.get(emo,    emo),
        "intensity" : RAVDESS_INTENSITY.get(inten, inten),
        "transcript": RAVDESS_STATEMENT.get(stmt,  f"statement_{stmt}"),
        "repetition": rep,
        "actor_id"  : actor_id,
        "gender"    : "male" if actor_id % 2 == 1 else "female",
        "emo_colour": RAVDESS_EMO_COLOUR.get(RAVDESS_EMOTION.get(emo, ""), "#E2E8F0"),
    }

meta = parse_ravdess(WAV_PATH)
if meta:
    print("RAVDESS file detected!")
    for k, v in meta.items():
        print(f"  {k:<12}: {v}")
    transcript  = meta["transcript"]
    emotion_lbl = meta["emotion"].upper()
    channel_lbl = meta["channel"].upper()
    emo_colour  = meta["emo_colour"]
    gender      = meta["gender"].upper()
    actor_id    = meta["actor_id"]
else:
    print("Non-RAVDESS file — using generic labels.")
    transcript  = "Unknown transcript"
    emotion_lbl = "UNKNOWN"
    channel_lbl = "SPEECH"
    emo_colour  = "#E2E8F0"
    gender      = "UNKNOWN"
    actor_id    = 0

known_words = transcript.split()
print(f"\nTranscript : \"{transcript}\"  ({len(known_words)} words)")


## Cell 4 — Load audio & detect word boundaries

In [ ]:
# ── 4a. Load original WAV at full resolution (48 kHz for display) ─────────────
print("Loading audio…")
sr48, raw48 = wavfile.read(WAV_PATH)
if raw48.ndim == 2:
    raw48 = raw48.mean(axis=1)
audio48 = raw48.astype(np.float32)
pk = np.abs(audio48).max()
if pk > 0:
    audio48 /= pk
audio_duration = len(audio48) / sr48
print(f"  Original : {sr48} Hz  |  {audio_duration:.3f}s  |  {len(audio48)} samples")

# ── 4b. Resample to 16 kHz for lightweight VAD ───────────────────────────────
subprocess.run(
    ["ffmpeg", "-y", "-i", WAV_PATH, "-ac", "1", "-ar", "16000", WAV_16K],
    capture_output=True, check=True
)
audio_sr, raw16 = wavfile.read(WAV_16K)
audio16 = raw16.astype(np.float32)
pk16 = np.abs(audio16).max()
if pk16 > 0:
    audio16 /= pk16
print(f"  Resampled: {audio_sr} Hz  |  {len(audio16)/audio_sr:.3f}s  |  {len(audio16)} samples")

# ── 4c. GIF dimensions & frame count ─────────────────────────────────────────
W, H     = OUTPUT_SIZE
n_frames = math.ceil(audio_duration * GIF_FPS)
os.makedirs(OUT_DIR, exist_ok=True)
print(f"  GIF frames: {n_frames} @ {GIF_FPS} fps  ({FRAME_MS} ms/frame)")

# ── 4d. Short-time energy VAD (8 ms frames, 3 ms hop) ────────────────────────
print("\nRunning short-time energy VAD…")
FL = int(0.008 * audio_sr)
HL = int(0.003 * audio_sr)
energies, times_vad = [], []
for i in range(0, len(audio16) - FL, HL):
    energies.append(float(np.sqrt(np.mean(audio16[i:i+FL]**2))))
    times_vad.append(i / audio_sr)
energies  = np.array(energies)
times_vad = np.array(times_vad)

# Smooth with 12 ms window
K_smooth = max(1, int(0.012 / (HL / audio_sr)))
smoothed  = np.convolve(energies, np.ones(K_smooth) / K_smooth, mode="same")

# Locate speech region (coarse voiced-segment detection)
threshold_coarse = smoothed.max() * 0.04
voiced            = smoothed > threshold_coarse
trans             = np.diff(voiced.astype(int))
onsets_c  = times_vad[np.where(trans ==  1)[0]]
offsets_c = times_vad[np.where(trans == -1)[0]]
regions   = list(zip(onsets_c[:len(offsets_c)], offsets_c))
# Merge gaps < 80 ms
merged = [list(regions[0])] if regions else [[0.0, audio_duration]]
for s, e in regions[1:]:
    if s - merged[-1][1] < 0.08:
        merged[-1][1] = e
    else:
        merged.append([s, e])
speech_s = float(merged[0][0])
speech_e = float(merged[-1][1])
print(f"  Speech window: {speech_s:.3f}s – {speech_e:.3f}s  ({speech_e-speech_s:.3f}s)")

# ── 4e. Peak-midpoint word boundary detection ─────────────────────────────────
n_words = len(known_words)
mask  = (times_vad >= speech_s) & (times_vad <= speech_e)
t_sp  = times_vad[mask]
e_sp  = smoothed[mask]

peaks_idx, _ = find_peaks(
    e_sp,
    distance=int(0.10 / (HL / audio_sr)),
    prominence=e_sp.max() * 0.05,
)
peak_times = t_sp[peaks_idx]
peak_vals  = e_sp[peaks_idx]

if len(peak_times) >= n_words:
    # Take n_words most prominent peaks, sorted by time
    top_n   = np.argsort(peak_vals)[-n_words:]
    top_sorted = np.sort(peak_times[top_n])
    boundaries = [(top_sorted[i] + top_sorted[i+1]) / 2
                  for i in range(n_words - 1)]
    word_starts = [speech_s] + boundaries
    word_ends   = boundaries + [speech_e]
else:
    # Fallback: proportional split across the speech window
    span = speech_e - speech_s
    word_starts = [speech_s + i / n_words * span for i in range(n_words)]
    word_ends   = [speech_s + (i+1) / n_words * span for i in range(n_words)]

word_timestamps = [
    {"word": w, "start": float(ws), "end": float(we)}
    for w, ws, we in zip(known_words, word_starts, word_ends)
]

print("\nWord timestamps:")
for wt in word_timestamps:
    print(f"  {wt['word']:12s}  {wt['start']:.3f}s – {wt['end']:.3f}s  "
          f"({wt['end']-wt['start']:.3f}s)")


## Cell 5 — Helper functions

In [ ]:
def hex_to_rgb(h):
    """'#RRGGBB' → (R, G, B) ints."""
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

def hex_to_float(h):
    """'#RRGGBB' → (r, g, b) floats for matplotlib."""
    return tuple(c / 255 for c in hex_to_rgb(h))

def best_font(paths, size):
    """Load the first available TrueType font, fallback to default."""
    for p in paths:
        if os.path.exists(p):
            return ImageFont.truetype(p, size)
    return ImageFont.load_default()

def active_word_at(t):
    """Return the word label being spoken at time t, or None."""
    for wt in word_timestamps:
        if wt["start"] <= t <= wt["end"]:
            return wt["word"]
    return None

print("Helpers defined.")


## Cell 6 — GIF 1: Scrolling audio waveform

In [ ]:
print(f"[GIF 1] Rendering {n_frames} waveform frames @ {GIF_FPS} fps …")
t0 = time.time()

dpi   = 100
bg_f  = hex_to_float(BG_COLOR)
wv_f  = hex_to_float(WAVE_COLOR)
acc_f = hex_to_float(ACCENT_COLOR)
txt_f = hex_to_float(TEXT_FG_COLOR)
hi_f  = hex_to_float(HIGHLIGHT_COLOR)
emo_f = hex_to_float(emo_colour)

# Pre-compute downsampled full waveform for background layer (48 kHz)
full_t48 = np.linspace(0, audio_duration, len(audio48))
step_bg  = max(1, len(audio48) // 3000)
ft_ds    = full_t48[::step_bg]
fa_ds    = audio48[::step_bg]

gif1_frames = []
for fi in range(n_frames):
    t_now       = fi / GIF_FPS
    active_wrd  = active_word_at(t_now)

    # 600 ms rolling window around the playhead
    win_s = max(0, t_now - 0.55)
    win_e = min(t_now + 0.05, audio_duration)
    smp_s = int(win_s * sr48)
    smp_e = int(win_e * sr48)
    seg   = audio48[smp_s:smp_e]
    t_seg = np.linspace(win_s, win_e, max(len(seg), 1))

    fig, ax = plt.subplots(figsize=(W / dpi, H / dpi), dpi=dpi)
    fig.patch.set_facecolor(bg_f)
    ax.set_facecolor(bg_f)

    # ── Background full waveform (dimmed) ────────────────────────────────────
    ax.plot(ft_ds, fa_ds, color=wv_f, linewidth=0.35, alpha=0.14)

    # ── Word-region shading + labels (always visible) ────────────────────────
    for wt in word_timestamps:
        ws, we   = wt["start"], wt["end"]
        is_active = (wt["word"] == active_wrd)
        ax.axvspan(ws, we, color=hi_f,
                   alpha=0.22 if is_active else 0.05, zorder=1)
        ax.text((ws + we) / 2, -0.82, wt["word"],
                ha="center", va="bottom", fontsize=7.5,
                color=hi_f,
                alpha=0.95 if is_active else 0.38,
                fontweight="bold" if is_active else "normal")

    # ── Rolling window waveform (bright) ─────────────────────────────────────
    step_w = max(1, len(seg) // 700)
    if len(seg) > 0:
        ax.plot(t_seg[::step_w], seg[::step_w],
                color=wv_f, linewidth=1.4, alpha=0.96)
        ax.fill_between(t_seg[::step_w], seg[::step_w], 0,
                        color=wv_f, alpha=0.18)

    # ── Playhead ─────────────────────────────────────────────────────────────
    ax.axvline(x=t_now, color=acc_f, linewidth=1.9, alpha=0.9, zorder=6)

    # ── Axes styling ──────────────────────────────────────────────────────────
    ax.set_xlim(0, audio_duration)
    ax.set_ylim(-1.15, 1.15)
    for sp in ax.spines.values():
        sp.set_edgecolor(acc_f)
        sp.set_linewidth(0.6)
    ax.tick_params(colors=txt_f, labelsize=7.5)
    ax.set_xlabel("Time (s)", fontsize=9, color=txt_f)
    ax.set_ylabel("Amplitude", fontsize=9, color=txt_f)
    title = (f'"{transcript}"   —   {emotion_lbl} / {channel_lbl}   '
             f'·   Actor {actor_id} ({gender})   ·   t = {t_now:.3f}s')
    ax.set_title(title, color=txt_f, fontsize=8, pad=5)

    # ── Progress bar (top strip, emotion-coloured) ────────────────────────────
    prog = min(t_now / audio_duration, 1.0)
    ax.axhline(y=1.07, xmin=0, xmax=1,
               color=hex_to_float("#1E293B"), linewidth=5, solid_capstyle="butt")
    ax.axhline(y=1.07, xmin=0, xmax=prog,
               color=emo_f, linewidth=5, solid_capstyle="butt")

    # ── Emotion badge ─────────────────────────────────────────────────────────
    ax.text(audio_duration * 0.975, 0.93, emotion_lbl,
            ha="right", va="top", fontsize=8, color=emo_f,
            fontweight="bold", transform=ax.transData,
            bbox=dict(boxstyle="round,pad=0.25", facecolor=bg_f,
                      edgecolor=emo_f, linewidth=0.8))

    fig.tight_layout(pad=0.40)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=dpi)
    buf.seek(0)
    gif1_frames.append(
        Image.open(buf).convert("RGB").resize((W, H), Image.LANCZOS).copy()
    )
    plt.close(fig)

    if (fi + 1) % 30 == 0 or fi == n_frames - 1:
        print(f"  frame {fi+1}/{n_frames}   t={t_now:.2f}s   "
              f"elapsed={time.time()-t0:.1f}s")

gif1_frames[0].save(
    OUT_GIF1, save_all=True, append_images=gif1_frames[1:],
    loop=0, duration=FRAME_MS, optimize=False
)
print(f"\n  ✓  {OUT_GIF1}")
print(f"     {os.path.getsize(OUT_GIF1) // 1024} KB  |  "
      f"{len(gif1_frames)} frames  |  {time.time()-t0:.1f}s total")


## Cell 7 — GIF 2: Word-by-word text display

In [ ]:
print(f"[GIF 2] Rendering {n_frames} text frames @ {GIF_FPS} fps …")
t0 = time.time()

font_word  = best_font([
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"], 36)
font_small = best_font([
    "/usr/share/fonts/truetype/dejavu/DejaVuSansMono-Bold.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf"], 11)
font_label = best_font([
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"], 11)

bg_rgb  = hex_to_rgb(BG_COLOR)
acc_rgb = hex_to_rgb(ACCENT_COLOR)
hi_rgb  = hex_to_rgb(HIGHLIGHT_COLOR)
dim_rgb = (50, 70, 95)       # not yet spoken
spo_rgb = (100, 140, 175)    # already spoken
emo_rgb = hex_to_rgb(emo_colour)

GAP     = 18   # px between words
MARGIN  = 20
TITLE_H = 36
STRIP_H = 30   # mini-waveform strip height

# Pre-compute word pixel widths
_dummy = ImageDraw.Draw(Image.new("RGB", (W, H)))
word_pxw = {
    wt["word"]: _dummy.textbbox((0, 0), wt["word"], font=font_word)[2]
    for wt in word_timestamps
}
total_row_px = sum(word_pxw.values()) + GAP * (n_words - 1)
x_start      = max(MARGIN, (W - total_row_px) // 2)
y_word       = TITLE_H + STRIP_H + 18     # below title bar + mini-waveform
y_bars       = y_word + 72                # per-word progress bars
y_ruler      = H - 42                     # timeline ruler

gif2_frames = []
for fi in range(n_frames):
    t_now = fi / GIF_FPS

    img  = Image.new("RGB", (W, H), bg_rgb)
    draw = ImageDraw.Draw(img)

    # ── Title bar ─────────────────────────────────────────────────────────────
    draw.rectangle([(0, 0), (W, TITLE_H)], fill=(14, 30, 52))
    draw.text(
        (MARGIN, 10),
        f"RAVDESS  ·  {emotion_lbl}  ·  {channel_lbl}  ·  "
        f"Actor {actor_id} ({gender})  ·  t = {t_now:.3f}s",
        fill=acc_rgb, font=font_small,
    )
    # Emotion badge
    bb   = draw.textbbox((0, 0), emotion_lbl, font=font_small)
    bw   = bb[2] - bb[0]
    draw.rectangle([(W - bw - 18, 8), (W - 8, TITLE_H - 8)],
                   outline=emo_rgb, width=1)
    draw.text((W - bw - 13, 10), emotion_lbl, fill=emo_rgb, font=font_small)

    # ── Mini waveform strip ───────────────────────────────────────────────────
    strip_y = TITLE_H + 2
    strip_w = W - 2 * MARGIN
    draw.rectangle([(MARGIN, strip_y), (W - MARGIN, strip_y + STRIP_H)],
                   fill=(16, 34, 58))
    seg_s = max(0, int((t_now - 0.40) * sr48))
    seg_e = min(len(audio48), int((t_now + 0.05) * sr48))
    mini  = audio48[seg_s:seg_e]
    if len(mini) > 1:
        step_m = max(1, len(mini) // strip_w)
        mini_ds = mini[::step_m]
        wv_rgb  = hex_to_rgb(WAVE_COLOR)
        for xi, amp in enumerate(mini_ds[:strip_w]):
            bh = max(1, int(abs(amp) * (STRIP_H // 2 - 2)))
            cy = strip_y + STRIP_H // 2
            draw.rectangle(
                [(MARGIN + xi, cy - bh), (MARGIN + xi + 1, cy + bh)],
                fill=wv_rgb,
            )
    # Playhead in strip
    ph_x = MARGIN + int(min(t_now / audio_duration, 1.0) * strip_w)
    draw.rectangle([(ph_x - 1, strip_y), (ph_x + 1, strip_y + STRIP_H)],
                   fill=acc_rgb)

    # ── Word rendering ────────────────────────────────────────────────────────
    x = x_start
    for wt in word_timestamps:
        ws, we   = wt["start"], wt["end"]
        active   = ws <= t_now <= we
        past     = t_now > we
        colour   = hi_rgb if active else (spo_rgb if past else dim_rgb)
        ww       = word_pxw[wt["word"]]

        # Soft glow behind active word
        if active:
            for dx, dy in [(-2, 0), (2, 0), (0, -2), (0, 2)]:
                draw.text((x + dx, y_word + dy), wt["word"],
                          fill=(110, 82, 12), font=font_word)

        draw.text((x, y_word), wt["word"], fill=colour, font=font_word)

        # Underline for active word
        if active:
            bb = draw.textbbox((x, y_word), wt["word"], font=font_word)
            draw.rectangle(
                [(bb[0], bb[3] + 3), (bb[2], bb[3] + 7)], fill=hi_rgb
            )

        # ── Per-word fill bar ─────────────────────────────────────────────────
        BH = 24
        draw.rectangle([(x, y_bars), (x + ww, y_bars + BH)], fill=(20, 40, 65))
        if t_now >= we:
            frac = 1.0
        elif t_now >= ws:
            frac = (t_now - ws) / max(we - ws, 1e-6)
        else:
            frac = 0.0
        if frac > 0:
            bc = hi_rgb if frac < 1.0 else spo_rgb
            draw.rectangle(
                [(x, y_bars), (x + int(ww * frac), y_bars + BH)], fill=bc
            )
        draw.text((x + 3, y_bars + BH // 2 - 5),
                  f"{we - ws:.2f}s", fill=(155, 185, 205), font=font_label)

        x += ww + GAP

    # ── Timeline ruler ────────────────────────────────────────────────────────
    ruler_w = W - 2 * MARGIN
    draw.rectangle([(MARGIN, y_ruler), (MARGIN + ruler_w, y_ruler + 4)],
                   fill=(24, 44, 68))
    # Word spans (accent-coloured blocks)
    for wt in word_timestamps:
        rx_s = MARGIN + int(wt["start"] / audio_duration * ruler_w)
        rx_e = MARGIN + int(wt["end"]   / audio_duration * ruler_w)
        draw.rectangle([(rx_s, y_ruler - 2), (rx_e, y_ruler + 6)],
                       fill=acc_rgb)
    # Playhead (diamond)
    rx = MARGIN + int(min(t_now / audio_duration, 1.0) * ruler_w)
    draw.polygon(
        [(rx, y_ruler - 8), (rx + 5, y_ruler + 1),
         (rx, y_ruler + 10), (rx - 5, y_ruler + 1)],
        fill=hi_rgb,
    )
    draw.text((MARGIN, y_ruler + 9), "0.0s", fill=dim_rgb, font=font_label)
    draw.text((W - MARGIN - 32, y_ruler + 9), f"{audio_duration:.2f}s",
              fill=dim_rgb, font=font_label)

    gif2_frames.append(img)

    if (fi + 1) % 30 == 0 or fi == n_frames - 1:
        print(f"  frame {fi+1}/{n_frames}   t={t_now:.2f}s   "
              f"elapsed={time.time()-t0:.1f}s")

gif2_frames[0].save(
    OUT_GIF2, save_all=True, append_images=gif2_frames[1:],
    loop=0, duration=FRAME_MS, optimize=False
)
print(f"\n  ✓  {OUT_GIF2}")
print(f"     {os.path.getsize(OUT_GIF2) // 1024} KB  |  "
      f"{len(gif2_frames)} frames  |  {time.time()-t0:.1f}s total")


## Cell 8 — Summary & first-frame preview

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print("─" * 60)
print("OUTPUT SUMMARY")
print("─" * 60)
total_kb = 0
for path, label in [(OUT_GIF1, "GIF1 – Waveform"),
                     (OUT_GIF2, "GIF2 – Text")]:
    kb = os.path.getsize(path) // 1024
    total_kb += kb
    g  = Image.open(path)
    nf = getattr(g, "n_frames", 1)
    print(f"  {label:<20}  {kb:>5} KB   {nf} frames   {FRAME_MS} ms/frame")
print(f"  {'TOTAL':<20}  {total_kb:>5} KB")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, path, title in zip(
    axes,
    [OUT_GIF1, OUT_GIF2],
    ["GIF 1 — Audio Waveform", "GIF 2 — Text Display"],
):
    ax.imshow(np.array(Image.open(path)))
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.suptitle(
    f'{os.path.basename(WAV_PATH)}  ·  "{transcript}"  ·  {emotion_lbl}',
    fontsize=9, y=1.01,
)
plt.tight_layout()
plt.savefig("preview_first_frames.png", dpi=90, bbox_inches="tight")
plt.show()
print("\nFirst-frame preview → preview_first_frames.png")


## Cell 9 — Play GIFs inline  *(Jupyter only)*

In [ ]:
from IPython.display import HTML
import base64

def gif_inline(path, label, width=640):
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode()
    return (
        f'<div style="display:block;margin:10px 0;text-align:left">'
        f'<p style="font-family:monospace;font-size:12px;color:#38BDF8;margin:0 0 4px">'
        f'{label}</p>'
        f'<img src="data:image/gif;base64,{data}" width="{width}" '
        f'style="border:1px solid #38BDF8;border-radius:4px;display:block"/>'
        f'</div>'
    )

HTML(
    '<div style="background:#0D1B2A;padding:16px;border-radius:6px">'
    + gif_inline(OUT_GIF1,
                 f"GIF 1 — Audio Waveform  |  {n_frames} frames  |  ~{GIF_FPS} fps")
    + gif_inline(OUT_GIF2,
                 f'GIF 2 — Transcript: "{transcript}"  |  {emotion_lbl}  |  '
                 f"Actor {actor_id} ({gender})")
    + "</div>"
)
